In [0]:
CREATE TABLE if not EXISTS project.silver.sales_delta_partition
(
  id INT,
  order_date DATE,
  amount DOUBLE,
  country STRING,
  year INT,
  month INT
)
USING DELTA
PARTITIONED BY (year, month);


In [0]:
INSERT INTO project.silver.sales_delta_partition VALUES
(1, '2025-01-05', 50000, 'India', 2025, 1),
(2, '2025-01-12', 60000, 'USA', 2025, 1),
(3, '2025-02-10', 55000, 'India', 2025, 2),
(4, '2025-03-15', 70000, 'UK', 2025, 3),
(5, '2025-03-15', 70000, 'UK', 2025, 4),
(4, '2025-03-15', 70000, 'UK', 2025, 5)
;


num_affected_rows,num_inserted_rows
6,6


In [0]:
desc table extended project.silver.sales_delta_partition

-- Only folder year=2024/month=2 is scanned → super fast. 

col_name,data_type,comment
id,int,null
order_date,date,null
amount,double,null
country,string,null
year,int,null
month,int,null
# Partition Information,,
# col_name,data_type,comment
year,int,null
month,int,null


In [0]:
-- With Partitioning Only folder year=2024/month=2 is scanned → super fast.
SELECT SUM(amount)
FROM project.silver.sales_delta_partition
WHERE year = 2024 AND month = 2;


sum(amount)
155950.0


In [0]:
-- Changing Partition Columns (Repartitioning)
-- Delta does not allow altering partition columns directly.
-- To change partitioning:



In [0]:
%python
df = spark.read.table("project.silver.sales_delta_partition")

In [0]:
%python
df.write.format("delta") \
  .partitionBy("country") \
  .mode("overwrite") \
  .saveAsTable("project.silver.sales_delta_repartitioned")


In [0]:
describe table extended project.silver.sales_delta_repartitioned

col_name,data_type,comment
id,int,null
order_date,date,null
amount,double,null
country,string,null
year,int,null
month,int,null
# Partition Information,,
# col_name,data_type,comment
country,string,null
,,


In [0]:
CREATE OR REPLACE TABLE project.silver.staging_sales (
  id INT,
  order_date DATE,
  amount DOUBLE,
  country STRING,
  year INT,
  month INT
);

INSERT INTO project.silver.staging_sales VALUES
(101, '2024-01-03', 4500, 'India', 2024, 1),
(102, '2024-01-07', 5200, 'USA', 2024, 1),
(103, '2024-01-11', 6100, 'UK', 2024, 1),
(104, '2024-02-02', 4800, 'India', 2024, 2),
(105, '2024-02-14', 5800, 'Germany', 2024, 2),
(106, '2024-02-20', 6900, 'USA', 2024, 2),
(107, '2024-03-05', 7200, 'India', 2024, 3),
(108, '2024-03-10', 5300, 'Japan', 2024, 3),
(109, '2024-03-25', 8100, 'UK', 2024, 3),
(110, '2024-04-01', 8900, 'USA', 2024, 4),
(111, '2024-04-08', 6500, 'India', 2024, 4),
(112, '2024-04-21', 7700, 'France', 2024, 4),
(113, '2024-05-04', 5600, 'Germany', 2024, 5),
(114, '2024-05-12', 6300, 'USA', 2024, 5),
(115, '2024-05-22', 7400, 'UK', 2024, 5),
(116, '2024-06-03', 9050, 'India', 2024, 6),
(117, '2024-06-14', 8200, 'Japan', 2024, 6),
(118, '2024-06-18', 9900, 'USA', 2024, 6),
(119, '2024-06-27', 10200, 'France', 2024, 6),
(120, '2024-06-30', 11300, 'India', 2024, 6);


num_affected_rows,num_inserted_rows
20,20


In [0]:
INSERT INTO project.silver.sales_delta_partition
PARTITION (year = 2024, month = 2)
SELECT id, order_date, amount, country
FROM project.silver.staging_sales;


num_affected_rows,num_inserted_rows
20,20


In [0]:
-- Partition Pruning (skipping folders not require for query)
-- Delta Lake automatically performs partition prunin

In [0]:
-- read the data where month =2
SELECT *
FROM project.silver.sales_delta_partition
WHERE month = 2;

id,order_date,amount,country,year,month
101,2024-01-03,4500.0,India,2024,2
102,2024-01-07,5200.0,USA,2024,2
103,2024-01-11,6100.0,UK,2024,2
104,2024-02-02,4800.0,India,2024,2
105,2024-02-14,5800.0,Germany,2024,2
106,2024-02-20,6900.0,USA,2024,2
107,2024-03-05,7200.0,India,2024,2
108,2024-03-10,5300.0,Japan,2024,2
109,2024-03-25,8100.0,UK,2024,2
110,2024-04-01,8900.0,USA,2024,2
